# Genetic Algorithm Test
The whole point of this is to test if we can do good optimisation of measurements to limbs using gradient free methods or not. 

In [5]:
from SSM_Driver import LegMeasurementDataset, measure, MeasurementLoss
import numpy as np
import torch

In [6]:
config = {
    "lr": 1e-3,
    "eta_min": 1e-6,
    "batch_size": 1,
    "log": True,
    "seed": 42,
    "epochs":200,
    "scale": True,
    "normalise": False,
    "relativise": False,
}
dtype = torch.float64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset = LegMeasurementDataset(measure, batch_size=config["batch_size"], scale=config["scale"], normalise=config["normalise"], relativise=config["relativise"], dtype=dtype, device="cpu")
dataloader = torch.utils.data.DataLoader(dataset, config["batch_size"], shuffle=False, num_workers=0)
component_transforms = torch.load("./data_components/scaled_component_transforms.pt").to(dtype).to(device)

loss = MeasurementLoss(dataset)
# loss(preds, components)

In [7]:
def step(x, y, func, lambda_=1E-3, delta=1E-6):
    # x.shape = (batch, components)
    # func(x).shape = (batch, measures)
    # y.shape = (batch, measures)
    with torch.no_grad():
        deltas = torch.eye(x.shape[1], device=x.device, dtype=x.dtype)*delta
        pos = func((x[:, None] + deltas).reshape((x.shape[0]*x.shape[1], x.shape[1]))).reshape((x.shape[0], y.shape[1], x.shape[1]))
        neg = func((x[:, None] - deltas).reshape((x.shape[0]*x.shape[1], x.shape[1]))).reshape((x.shape[0], y.shape[1], x.shape[1]))
        func_x = func(x)

        J_approx = -(pos - neg) / (2*delta)

        # Compute residuals: (batch_size, n_measures, 1)
        residuals = (y - func_x).unsqueeze(-1)

        # JTJ: (batch_size, n_components, n_components)
        JTJ = torch.bmm(J_approx.transpose(1, 2), J_approx)

        # Damping (use scalar, e.g. 1e-3)
        damping = torch.diag_embed(torch.diagonal(JTJ, dim1=1, dim2=2))

        # J^T * residuals: (batch_size, n_components, 1)
        JTr = torch.bmm(J_approx.transpose(1, 2), residuals)

        # LM step direction: (batch_size, n_components)
        x_new = x + torch.linalg.solve(JTJ + lambda_ * damping, JTr).squeeze(-1)
        new_residuals = (y - func(x_new))
        while torch.sum(residuals**2) < torch.sum(new_residuals**2):
            lambda_ *= 10
            x_new = x + torch.linalg.solve(JTJ + lambda_ * damping, JTr).squeeze(-1)
            new_residuals = (y - func(x_new))

            if lambda_ > 1E10:
                raise ValueError()

        return x_new, new_residuals, lambda_ / 10

In [ ]:
import pygad
import csv
import os
class FitnessFunction:
    def __init__(self, components, loss, transforms):
        self.components = components
        self.loss = loss
        self.transforms = transforms

    def calc(self, instance, solution, idx):
        # solution is a 1D numpy array, convert to torch tensor
        pred_components = torch.tensor(solution, dtype=dtype, device=device).unsqueeze(0)
        pred_components = pred_components * self.transforms[1:] + self.transforms[:1]

        # calculate loss (negative because pygad maximizes fitness)
        loss_value = self.loss(pred_components, self.components).item()
        return -loss_value
    
    

for measures, components in dataloader:
    fitness_func = FitnessFunction(components, loss, component_transforms).calc
    num_genes = components.shape[1]
    gene_space = {'low': -3, 'high': 3}  # adjust bounds as needed

    ga_instance = pygad.GA(
        num_generations=50,
        num_parents_mating=4,
        fitness_func=fitness_func,
        sol_per_pop=16,
        num_genes=num_genes,
        gene_space=gene_space,
        parent_selection_type="sss",
        keep_parents=2,
        mutation_percent_genes=20,
        crossover_type="single_point",
        mutation_type="random"
    )

    ga_instance.run()
    best_solution, best_solution_fitness, _ = ga_instance.best_solution()

    # Refine with 10 steps of the step function
    x = torch.tensor(best_solution, dtype=dtype, device=device).unsqueeze(0)
    y = measures
    func = dataset.get_measures
    lambda_ = 1e-3
    try:
        for _ in range(10):
            x, _, lambda_ = step(x, y, func, lambda_=lambda_)
    except ValueError:
        pass
    refined_solution = x.squeeze(0).detach()

    # Use refined_solution for further evaluation
    solution_tensor = refined_solution * component_transforms[1:] + component_transforms[:1]
    solution_difference = torch.abs(dataset.get_measures(solution_tensor) - dataset.get_measures(components))
    solution_tensor = torch.tensor(best_solution, dtype=dtype, device=device) * component_transforms[1:] + component_transforms[:1]
    solution_difference = torch.abs(dataset.get_measures(solution_tensor) - dataset.get_measures(components))

    file_exists = os.path.isfile("GA_results.csv")
    with open("GA_results.csv", mode="a", newline="") as file:
        writer = csv.writer(file)
        if not file_exists:
            writer.writerow(
            [f"measures_{i+1}" for i in range(measures.shape[1])]
            + [f"components_{i+1}" for i in range(components.shape[1])]
            + [f"solution_{i+1}" for i in range(len(best_solution))]
            + ["fitness"]
            + [f"solution_difference_{i+1}" for i in range(solution_difference.shape[1])]
            )
        writer.writerow(
            measures.cpu().numpy().flatten().tolist()
            + components.cpu().numpy().flatten().tolist()
            + best_solution.tolist()
            + [float(best_solution_fitness)]
            + solution_difference.cpu().numpy().flatten().tolist()
        )
    # print(f"Best Solution (components): {best_solution}")
    # print(f"Best Solution Fitness (negative loss): {best_solution_fitness}")
    print(f"Solution difference: {solution_difference}\n")

Solution difference: tensor([[7.0138, 1.6694, 3.1295, 2.0923, 2.8767, 2.7637, 3.3850]],
       dtype=torch.float64)

